# 00 — Setup, model inspection and library test

This is an example series on **ablation** — the idea is to try to answer the question "what happens if I remove this?" iteratively from a LLM model.

The demo will implement three different ablation types and compare results between them.

Same underlying idea each time — delete a piece, see what breaks — applied to a transformer's internals, its learned representations, and the application built on top of it.

All three notebooks use the same small, ungated model so nothing here needs a Hugging Face login or a GPU: **`Qwen/Qwen2.5-0.5B-Instruct`** (~1 GB, runs comfortably on a Colab/laptop CPU).

This notebook just installs the package and confirms the model loads and generates — a quick sanity check before relying on it for the rest of the series.

In [1]:
# If you're running this in Colab, uncomment the line below (point it at
# your fork/clone of the repo). If you cloned the repo locally and already
# ran `uv sync --extra notebook` (or `pip install -e ".[notebook]"`) in your
# virtualenv, you can skip this cell.

# %pip install -q "llm-ablation[notebook] @ git+https://github.com/<your-username>/llm-ablation.git"
#
# If running locally, use Astral's own UV package manager and just launch "uv sync" from the terminal

In [2]:
# import pytorch
import torch
# load local model load library
from llm_ablation.models import load_model

# load model
bundle = load_model()
print(f"model:    {bundle.name}")
print(f"device:   {bundle.device}")
print(f"layers:   {bundle.n_layers}")
print(f"heads:    {bundle.n_heads}")
print(f"head_dim: {bundle.head_dim}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model:    Qwen/Qwen2.5-0.5B-Instruct
device:   mps
layers:   24
heads:    14
head_dim: 64


In [3]:
# display the network via pytorch
print(bundle.model)
print(bundle.tokenizer)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

A quick generation to confirm everything actually works end to end:

In [4]:
# use the tokenizer to create an input tensor for the model
input_text: str = "Hello, how are you?"
input_tensor = bundle.tokenizer(input_text, return_tensor="pt")
print(input_tensor)

{'input_ids': [9707, 11, 1246, 525, 498, 30], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [5]:
# display model structure via torchinfo
from torchinfo import summary

# generate model summary
model_summary = summary(bundle.model, input_ids=input_tensor.get('input_ids'))

# display
print(model_summary)

Layer (type:depth-idx)                             Param #
Qwen2ForCausalLM                                   --
├─Qwen2Model: 1-1                                  --
│    └─Embedding: 2-1                              136,134,656
│    └─ModuleList: 2-2                             --
│    │    └─Qwen2DecoderLayer: 3-1                 14,912,384
│    │    └─Qwen2DecoderLayer: 3-2                 14,912,384
│    │    └─Qwen2DecoderLayer: 3-3                 14,912,384
│    │    └─Qwen2DecoderLayer: 3-4                 14,912,384
│    │    └─Qwen2DecoderLayer: 3-5                 14,912,384
│    │    └─Qwen2DecoderLayer: 3-6                 14,912,384
│    │    └─Qwen2DecoderLayer: 3-7                 14,912,384
│    │    └─Qwen2DecoderLayer: 3-8                 14,912,384
│    │    └─Qwen2DecoderLayer: 3-9                 14,912,384
│    │    └─Qwen2DecoderLayer: 3-10                14,912,384
│    │    └─Qwen2DecoderLayer: 3-11                14,912,384
│    │    └─Qwen2DecoderLayer: 3-1

In [6]:
# try inference
print(bundle.generate("What is the capital of France?", max_new_tokens=30))

The capital of France is Paris.


If that printed a sensible sentence, the local/cloud pytorch environment is correctly running.

In [7]:
# load machinery
from llm_ablation.mech import logit_diff, _first_token_id

# try logits calculation
correct_word: str = "Paris"
incorrect_word: str = "Milan"
# this gets fed to the model
prompt: str = "What is the capital of France?"


In [8]:
# perform inference
bundle.generate(prompt)

# try calculating logit difference between correct and incorrect answers
inputs = bundle.tokenizer(prompt, return_tensors="pt").to(bundle.device)
print(f"Input Tokens: {type(inputs)}, len: {len(inputs.get("input_ids")[0])}")

# get logits (probs) for all token positions
with torch.no_grad():
    out_logits = bundle.model(**inputs).logits[0,-1]
print(f"Logits: {type(out_logits)} - Len: {len(out_logits)}")

# calculate id for correct and incorrect tokens
correct_tok = _first_token_id(bundle, correct_word)
incorrect_tok = _first_token_id(bundle, incorrect_word)
# calculate logit difference
logit_difference: float = float(out_logits[correct_tok] - out_logits[incorrect_tok])
print(f"Logit Difference: {logit_difference}")

# do the same with the library
lg_diff = logit_diff(bundle, prompt, correct_word, incorrect_word)
print(f"Same, with library call: {lg_diff}")

Input Tokens: <class 'transformers.tokenization_utils_base.BatchEncoding'>, len: 7
Logits: <class 'torch.Tensor'> - Len: 151936
Logit Difference: 7.359375
Same, with library call: 7.359375
